# darts_forecast — the strategy: forecasts into trades, the baseline, the study, the pick

A Darts model (`model` in [`config.yaml`](./config.yaml): `gbm` | `gbm_clf` | `tide` | `tft`) forecasts each name's
return over the next `horizon` grid bars of `bar_minutes` (default: the next 5-minute bar). It runs walk-forward:
before every session (`retrain_minutes: 390`) it is refit on the last `train_days` sessions only, and each forecast
reads bars up to its own decision only. **This notebook takes those forecasts as given.** Choosing and checking the
model is [`darts_forecast_model.ipynb`](./darts_forecast_model.ipynb)'s job. The question here is the trading one:
traded with stops, risk-per-trade sizing and the model-health gate, in ONE shared \$10k wallet, after costs, does
the book earn more than the market exposure it runs?

**Which data each step reads** (the split lives in `research.WINDOWS`; Fig 1 draws it):

| window | dates | read here by |
|---|---|---|
| TUNE · warm-up | the first 21 sessions from 2025-06-18 | the daily inputs and the volatility scale only |
| TUNE · TRAIN | the `train_days` sessions before each retrain point, rolling | each model fit (TiDE / TFT: the last `val_days` of them are the VALIDATION slice for early stopping) |
| TUNE · scored | `optuna.score_from` (2025-09-16, the first fitted model) → 2026-01-31 | §3–§10: the backtest, the baseline, the study, the pick, the retraining cadence. Every forecast here is out of sample. |
| TEST | 2026-02-01 → 2026-04-30 | nothing by default. Its one-shot read is [`darts_forecast_diagnostics.ipynb`](./darts_forecast_diagnostics.ipynb); `READ_TEST = True` replays the frozen rules here (§11, §12) |
| HOLDOUT | 2026-05-01 → last stored bar | never loaded here (the diagnostics notebook, behind `READ_HOLDOUT`) |

| § | what |
|---|---|
| 0 | the knobs |
| 1 | the data and the walk-forward: the panel, the option inputs, the fit log, the split as a timeline |
| 2 | every knob: shipped value, sweep grid, Optuna range |
| 3 | one backtest of the shipped rules, warm from `score_from`: stats, equity, trades |
| 4 | **the baseline comparison**: the book against equal-weight buy-and-hold of the same names |
| 5 | from forecast to trade: the decision funnel, the health gate, trade return by forecast size, one name on one day |
| 6 | the Optuna study of the trading rule: one objective (`alpha_cagr`), in-process, on the forecasts cached in §1 |
| 7 | the pick among all admissible trials, alpha against each searched knob, the candidates file |
| 8 | the boundary check |
| 9 | the picked rules against the shipped rules and the baseline (TUNE) |
| 10 | does retraining pay? the same rules on forecasts retrained once, monthly, weekly, daily (TUNE) |
| 11 | the TEST replay of the frozen rules: only with `READ_TEST = True` |
| 12 | cost stress: 2× every cost, a 10/3 bp bar, pessimistic fills (TUNE; TEST too with `READ_TEST`) |
| 13 | checks and the scorecard |

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --all-extras  (Darts is the "darts" extra)
import functools
import importlib.util
import os
import warnings
from dataclasses import replace

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, optimize as opt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds what the darts_forecast notebooks share (loading, the option inputs, the
# forecast cache, the split, the backtest tables), charts.py the figure helpers and checks.py the
# pipeline checks; signals() never imports them.
_spec = importlib.util.spec_from_file_location(
    "darts_research", strategies.STRATEGIES_DIR / "darts_forecast" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
_spec = importlib.util.spec_from_file_location(
    "darts_charts", strategies.STRATEGIES_DIR / "darts_forecast" / "charts.py"
)
charts = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(charts)
_spec = importlib.util.spec_from_file_location(
    "darts_checks", strategies.STRATEGIES_DIR / "darts_forecast" / "checks.py"
)
ck = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ck)
strat, S = rs.STRAT, rs.SM  # THE loaded strategy: its module holds the in-process forecast cache
baseline = strategies.load("baseline")

# --- 0 · the knobs --------------------------------------------------------------------------------
FORECAST = {}  # forecast knobs to change from config.yaml, e.g. {"model": "tide"},
#     {"bar_minutes": 15} or {"retrain_minutes": 1950}. Each forecast config gets its own study
#     journal (STUDY) and its own cached walk-forward: gbm ~8 min on TUNE, tide / tft hours.
N_TRIALS = strat.optuna["n_trials"]  # §6 tops the study up TO this many trials, never past it
CADENCES = (0, 7800, 1950, 390)  # §10: retrain every N trading minutes (0 = once, 390 = daily);
#     each is its own cached walk-forward (60 = hourly costs ~50 min with gbm)
COST_BARS = [(10.0, 3.0)]  # §12: (quoted spread, slippage per leg) in bp, beside 2x costs
SYMBOL, DAY = None, None  # §5d: one name on one day; None = the most traded name, its busiest day
READ_TEST = False  # §11-§12: True loads TEST (2026-02-01 -> 2026-04-30) and replays the frozen
#     rules on it. Off: the one-shot TEST read is darts_forecast_diagnostics.ipynb's, and a knob
#     changed after reading TEST makes TEST one more tuning window
# Headless smoke runs only: DARTS_FORECAST_SMOKE_TRIALS=8 runs that many trials into a separate
# "-smoke" journal, so a test run never mixes into the real study.
SMOKE = int(os.environ.get("DARTS_FORECAST_SMOKE_TRIALS", "0"))

PRM = {**strat.params, **FORECAST}  # "the shipped rules" below: config.yaml (+ FORECAST)
assert strat.valid(PRM), f"config.yaml + FORECAST is not a valid combination: {FORECAST}"
run_strat = replace(strat, params=PRM)  # what every backtest and the study run
STUDY = strat.optuna["study"] + "".join(f"-{k}{v}" for k, v in sorted(FORECAST.items()))
STUDY += "-smoke" if SMOKE else ""
N_TRIALS = SMOKE or N_TRIALS
OBJECTIVES = opt.objectives_for(strat)  # one: alpha_cagr
TUNE, TEST, HOLDOUT = (rs.WINDOWS[k] for k in ("tune", "test", "holdout"))
FLOOR = strat.optuna.get("pick_min_trades_per_year", strat.optuna["min_trades_per_year"])
RULE = [k for k in PRM if k not in S["MODEL_KEYS"]]  # the trading-rule knobs
INIT_CASH = settings.backtest.init_cash
COLS = [
    "cagr",
    "alpha_cagr",
    "beta",
    "excess_cagr",
    "info_ratio",
    "volatility",
    "sharpe",
    "max_drawdown",
    "n_trades",
    "trades_per_year",
    "win_rate",
    "pct_profitable",
    "years",
]
print(
    f"study {STUDY} · objectives {list(OBJECTIVES)} · up to {N_TRIALS} trials · "
    f"READ_TEST {READ_TEST}"
)
rs.windows_table()

## 1 · The data and the walk-forward (Table 1a, Table 1b, Fig 1)

The panel holds the STORED 1-minute regular-session bars of the five names, from `data.start` through the last TUNE
session (`research.load`), with the option inputs attached from the free options store. On a half-day the stored
"regular" session runs past the 13:00 close; the strategy masks those minutes (`strategy.HALF_DAYS`). Nothing after
the TUNE end is loaded unless `READ_TEST = True`.

**The walk-forward is computed once and cached.** `research.prime_cache` loads it from `cache/`, or runs it and
saves it, and hands it to `signals()`. So every backtest and every study trial below re-scores the same forecasts
and never refits: a trial costs seconds. The cache key holds every forecast knob, the panel's full content (bars and
option inputs) and the version of `strategy.py`, so a changed knob, a backfilled bar or a newer options store means
a new walk-forward, never a stale one. The first run of a forecast config pays for it (gbm about 8 minutes here,
TiDE / TFT hours), and §10 pays once more for each retrain cadence not cached yet.

The printout says whether the option inputs are attached (§6 refuses to run the study without them). It then counts
every forecast, and the **scored** ones: those from `score_from` on whose outcome ends inside the session (the last
grid bar of each session has none).

* **Table 1a** the option inputs: which ones the model reads (`options`), their source, since when each has data,
  and its coverage of the scored sessions. Missing data enters the model as 0, plus one "has data" flag per source
  (`has_dolt_iv`, `has_dolt_hv`, `has_occ`, `has_oi`).
* **Table 1b** the fit log, first and last fits (numbered from 1). Each fit trains on the sessions `train_from` →
  `train_last`, then forecasts until the next retrain point.
* **Fig 1** the split as a timeline, with three sample fits: what each one trained on and what it forecast. A daily
  retrain makes about 95 fits on TUNE, each forecasting one session, so a fit's forecast span is one session wide.
  On the TUNE row, the TRAIN bar between the warm-up and the scored months is the first fit's window: no forecast
  exists before it. `gbm` has no VALIDATION slice (`val_days` is for TiDE / TFT early stopping); with, for example,
  `FORECAST = {"model": "tide", "val_days": 5}` the timeline draws it at the end of every TRAIN window.

In [ ]:
panel = rs.load(settings, engine)  # data.start -> the last TUNE session, options attached
fc = rs.prime_cache(panel, PRM)  # the walk-forward: from cache/, or run once and saved
log, inp, g = fc["log"], fc["inputs"], fc["inputs"]["g"]
SCORE_FROM = max(strat.optuna["score_from"], log["session"].iloc[0])  # the first fitted model
sess = pd.Index(sorted(set(panel.close.index.tz_convert(panel.tz).date)))
OPT_FRAMES = sorted(k for k in (panel.features or {}) if k.startswith("opt_"))
print(
    f"panel: {len(panel.close):,} 1-minute bars x {len(panel.symbols)} names "
    f"({', '.join(panel.symbols)}), {len(sess)} sessions {sess[0]} → {sess[-1]}"
)
print(
    f"option inputs attached: {inp['options_attached']} ({len(OPT_FRAMES)} opt_* frames); "
    f"the model reads options={PRM['options']!r}"
    + (
        ""
        if inp["options_attached"] or PRM["options"] == "none"
        else " -- !! NONE ATTACHED: every option input is 0 with its has_* flag off"
    )
)
print(
    f"walk-forward: {PRM['model']} on a {PRM['bar_minutes']}-minute grid, horizon "
    f"{PRM['horizon']}, {len(log)} fits (retrain every {PRM['retrain_minutes']} trading "
    f"minutes), {log['seconds'].sum() / 60:.1f} min of fitting, "
    f"{int(np.isfinite(fc['edge']).sum()):,} forecasts"
)
print(f"scored from {SCORE_FROM} (config optuna.score_from: {strat.optuna['score_from']})")
t_tune = rs.table(fc, (SCORE_FROM, TUNE[1]))  # the scored forecasts: outcome inside the session
sk = rs.summary(t_tune)
print(
    f"scored forecasts: {len(t_tune):,} on {t_tune['date'].nunique()} sessions; rank IC "
    f"{sk['ic']:+.4f} (t {sk['t']:+.2f}), hit {sk['hit']:.1%}, AUC {sk['auc']:.3f}, "
    f"MSE skill {sk['mse_skill']:+.4f}"
)

src_of = {n: src for src, names in S["OPTION_SOURCES"].items() for n in names}
read = set(S["OPTION_SET_SOURCES"][PRM["options"]])
start_ts = pd.Timestamp(SCORE_FROM, tz="UTC")
table1a = pd.DataFrame(
    {
        k.removeprefix("opt_"): {
            "source (has_* flag)": src_of.get(k.removeprefix("opt_"), "?"),
            "model input": src_of.get(k.removeprefix("opt_")) in read,
            "first session with data": str(panel.features[k].dropna(how="all").index.min())[:10],
            "coverage, scored TUNE": round(
                panel.features[k][panel.features[k].index >= start_ts].notna().to_numpy().mean(), 2
            ),
        }
        for k in OPT_FRAMES
    }
).T
display(table1a.sort_values("source (has_* flag)", kind="stable"))
ends = log if len(log) <= 6 else pd.concat([log.head(3), log.tail(3)])  # first and last fits
display(ends.assign(fit=lambda d: d["fit"] + 1).set_index("fit"))

In [ ]:
sd = pd.to_datetime(pd.Series(sess))
val = PRM["val_days"] if PRM["model"] in rs.SM["DEEP"] else 0  # gbm has no validation slice
warm = max(21, PRM["vol_days"] + 1)  # strategy.forecast's warm-up sessions
fig = go.Figure()

charts.seg(fig, "TUNE", sd.iloc[0], sd.iloc[warm - 1], plots._MUTED, "warm-up: inputs only")
# the first fit's window: no forecast exists before it
charts.fit_window(fig, "TUNE", log.iloc[0], val, sess)
charts.seg(fig, "TUNE", SCORE_FROM, sd.iloc[-1], plots.SERIES[0], "scored: §3-§10")
for k in sorted({0, len(log) // 2, len(log) - 1}):
    r = log.iloc[k]
    y = f"fit {r['fit'] + 1} · {r['session']}"
    charts.fit_window(fig, y, r, val, sess)
    nxt = log["session"].iloc[k + 1] if k + 1 < len(log) else sd.iloc[-1] + pd.Timedelta(days=1)
    charts.seg(fig, y, r["session"], nxt, plots.SERIES[7], "forecast (out of sample)")
charts.seg(
    fig,
    "TEST",
    TEST[0],
    TEST[1],
    plots.SERIES[2],
    "TEST: §11 with READ_TEST" if READ_TEST else "TEST: not loaded here (diagnostics)",
    dash=None if READ_TEST else "dot",
)
charts.seg(
    fig,
    "HOLDOUT",
    HOLDOUT[0],
    pd.Timestamp.today().normalize(),
    plots.SERIES[5],
    "HOLDOUT: never loaded here",
    dash="dot",
)
plots._base(
    fig, title="Fig 1 · the data split and three sample fits of the walk-forward", height=380
)
fig.update_yaxes(autorange="reversed", type="category")
fig.update_xaxes(type="date")
charts.below(fig, -0.15).show()

## 2 · Every knob — shipped value, sweep grid, Optuna range

| group | knobs | what they decide |
|---|---|---|
| forecaster | `model`, `bar_minutes`, `horizon`, `lookback`, `cov_lags`, `gbm_iter`, `n_epochs`, `device`, `seed` | which Darts model, on which grid, how far ahead, how much history it reads |
| training | `train_days`, `val_days`, `retrain_minutes`, `finetune_epochs`, `recency_half_life` | the rolling TRAIN window and the VALIDATION slice (tide / tft), the retrain cadence (390 = before every session, 0 = train once), fine-tuning, recency weighting |
| inputs | `options`, `use_daily`, `use_market`, `vol_days` | which input groups the model reads; the causal volatility scale behind every return input |
| trading rule | `sides`, `edge_bp`, `p_min`, `hold_bars`, `skip_open`, `skip_close`, `eod_exit` | when a forecast becomes a trade, how long it is held, flat by the close |
| risk | `stop_vol`, `risk_per_trade`, `max_positions` | the volatility stop, and the size: `min(1 / max_positions, risk_per_trade / stop)` of free cash |
| health gate | `health_days`, `health_min` | stand aside on days after the model's own recent forecasts stopped ranking the outcomes |

The forecaster, training and input knobs are fixed per study: `FORECAST` in §0 changes them and opens a new
journal. The study searches only the trading rule, because re-scoring cached forecasts costs seconds and a new
walk-forward costs minutes to hours. The table below is rendered from `config.yaml`: `this run` includes
`FORECAST`. The sweep grid is not run here; the study covers those knobs.

In [ ]:
pd.DataFrame(
    {
        "shipped (config.yaml)": pd.Series(strat.params, dtype=object),
        "this run": pd.Series(PRM, dtype=object),
        "sweep grid": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
        "Optuna range (§6)": pd.Series(
            {k: rs.fmt_space(s) for k, s in strat.optuna["space"].items()}
        ),
        "role": pd.Series({k: "forecast" if k in S["MODEL_KEYS"] else "trading rule" for k in PRM}),
        "doc": pd.Series(strat.docs),
    }
).loc[list(PRM)].fillna("—")

## 3 · One backtest of the shipped rules — warm from `score_from` (Table 3a–3c, Fig 3)

The engine rules are the same for every strategy in the repo:

* A decision is taken at a grid bar's last minute and **fills at the next 1-minute bar's open** (`fill: open`).
* **The position path.** Each decision asks to hold its side for `hold_bars` grid bars (0 = `horizon`). A new
  same-side decision extends the hold; an opposite one closes it, and the new side opens one minute later. The book is
  flat at the session's last real bar (12:59 on a half-day).
* **One shared \$10k wallet.** Each entry deploys `min(1 / max_positions, risk_per_trade / stop)` of the FREE cash,
  so a stop-out costs at most `risk_per_trade` of the free cash (gaps aside) and the book is never more than fully
  invested. Shorts lock their sale proceeds. The stop is `stop_vol` typical moves over the hold, about 0.3–0.9% on a
  5-minute hold of these names, so at `risk_per_trade: 0.001` the risk rule sets the size of most trades (a 0.6% stop
  deploys about 16% of the free cash, under the 20% slot). The line under Table 3b prints the stop distance and which
  of the two set each trade's size.
* **Costs at the real order size.** `order_notional: 2000` puts the \$1 minimum commission at 5 bp per leg, plus
  half the spread and the slippage: 14 bp per round trip (`research.round_trip_bp`, printed below).
* **Warm.** `signals()` runs on the whole panel, so the forecasts and the health gate have their history. The book
  and every metric start at `score_from`, the first session with a fitted model.

How to read Table 3a:

* `alpha_cagr` is the study's objective: the return left after the market exposure the book actually ran
  (`cagr − beta × buy-and-hold cagr`).
* `excess_cagr` is the plain difference to buy-and-hold. A book that is flat most of the day cannot win it by
  construction.
* `win_rate` counts trades; `pct_profitable` counts names.

**Table 3b** splits the trades by side (net return per trade in bp, and before costs ≈ net + one round trip).
**Table 3c** is the per-name view: a name's `total_return` is its contribution to the one shared book, not its own
return.

In [ ]:
# scored(params, ...): one warm backtest, the portfolio and bt.evaluate's numbers from ONE run
# (research.scored); by default the shipped rules on the TUNE panel, from SCORE_FROM
scored = functools.partial(rs.scored, strategy=run_strat, p=panel, start=SCORE_FROM, sett=settings)
fees, slip = bt.strategy_costs(run_strat, panel, settings)
RT_BP = rs.round_trip_bp(settings, panel)  # the engine's round trip at the $2,000 order size
RT = RT_BP / 1e4  # one round trip, as a fraction of the order
pf, st = scored({})
pf_base, st_base = scored({}, strategy=baseline)
floor = strat.optuna["min_trades_per_year"]
print(
    f"costs per leg: fees {1e4 * fees:.1f} bp (commission at ${strat.order_notional:,.0f} orders + "
    f"half the spread) + slippage {1e4 * slip:.1f} bp → one round trip {RT_BP:.1f} bp"
)
print(
    f"shipped rules: {st['n_trades']:,} trades, {st['trades_per_year']:,.0f}/yr over "
    f"{st['years']:.2f} years; the study's floor is {floor}/yr: "
    + ("clears it" if st["trades_per_year"] >= floor else "BELOW it (the study prunes such rules)")
)
display(pd.DataFrame({"shipped rules": st, "baseline (buy & hold)": st_base}).loc[COLS].round(4))
fig = plots.plot_equity(
    bt.equity_curve(pf, init_cash=INIT_CASH),
    title=f"Fig 3 · shipped rules: equity, fees paid vs realized P&L (from {SCORE_FROM})",
)
for note in fig.layout.annotations:  # a title holding two "$" amounts is typeset as TeX math
    note.text = note.text.replace("$", "&#36;")  # wherever MathJax runs; the entity prints "$"
charts.below(fig, -0.08).show()

In [ ]:
trades = bt.trade_records(pf)
trades["minutes"] = (trades["exit_ts"] - trades["entry_ts"]).dt.total_seconds() / 60
trades["return_bp"] = 1e4 * trades["return"]  # net of costs, signed as positioned
by_side = trades.groupby("direction").agg(
    trades=("pnl", "size"),
    win_rate=("pnl", lambda s: (s > 0).mean()),
    net_bp=("return_bp", "mean"),
    median_minutes=("minutes", "median"),
    pnl_usd=("pnl", "sum"),
)
by_side["before_costs_bp"] = by_side["net_bp"] + RT_BP
display(by_side.round(2))
# the stop and the size of each trade: the signal frames at its entry bar, as the engine reads them
sig0 = S["trade_frames"](panel, fc, **{k: PRM[k] for k in RULE})
r_in = panel.close.index.get_indexer(trades["entry_ts"])
c_in = pd.Index(panel.symbols).get_indexer(trades["symbol"])
trades["stop_bp"] = 1e4 * sig0["sl_stop"].to_numpy()[r_in, c_in]  # stop distance from the entry
trades["cash_frac"] = sig0["cash_frac"].to_numpy()[r_in, c_in]  # share of the free cash deployed
slot = 1.0 / PRM["max_positions"]
if len(trades):
    by_risk = trades["cash_frac"] < slot - 1e-12
    print(
        f"stop at the entry: median {trades['stop_bp'].median():.0f} bp "
        f"({trades['stop_bp'].min():.0f}-{trades['stop_bp'].max():.0f}); the size is "
        f"risk_per_trade / stop on {by_risk.mean():.0%} of trades (median "
        f"{trades.loc[by_risk, 'cash_frac'].median():.0%} of the free cash) and the "
        f"1/{PRM['max_positions']} slot on the rest, so a typical stop-out loses "
        f"{(trades['cash_frac'] * trades['stop_bp']).median() / 100:.2f}% of the free cash "
        f"(risk_per_trade: {PRM['risk_per_trade']:.2%})"
    )
display(bt.symbol_stats(pf).round(4))
trades.head(8).round(4)

## 4 · The baseline comparison (Fig 4, Table 4a, Table 4b)

Every strategy in the repo faces the same hurdle: `strategies/baseline`, equal-weight buy-and-hold of the same five
names, on the same panel, scored from the same first bar. `bt.active_stats` re-prices it at the strategy's own costs
and fill model. Read **alpha**, not the gap between the curves. The book is flat overnight and most of the day, so
its beta is near zero and it cannot out-earn a fully invested basket in a rising market. It must still earn more
than the exposure it did run.

* **Table 4a** the active statistics of the shipped book against that basket.
* **Table 4b** the same comparison month by month: an edge that lives in one month is a story, not an edge.

In [ ]:
charts.overlay(
    {"shipped rules": pf},
    f"Fig 4 · cumulative return from {SCORE_FROM}: the book vs buy-and-hold",
    pf_base,
    INIT_CASH,
).show()
ACTIVE = ["alpha_cagr", "beta", "excess_cagr", "info_ratio", "tracking_error"]
display(pd.DataFrame({"shipped rules": {m: st[m] for m in ACTIVE}}).T.round(4))
by_month = pd.DataFrame(
    {
        "shipped rules": rs.monthly(pf, INIT_CASH, panel.tz),
        "baseline": rs.monthly(pf_base, INIT_CASH, panel.tz),
    }
)
by_month["difference"] = by_month["shipped rules"] - by_month["baseline"]
by_month.round(4)

## 5 · From forecast to trade (Table 5a–5c, Fig 5b–5d)

The book is the forecast run through a few filters. Each part answers one question.

**5a · the funnel.** How many forecasts survive each gate of the rule, and what the book makes of the survivors.
The gates:

* the trading hours: no decision in the first `skip_open` or the last `skip_close` minutes, none whose hold would
  cross the close, none after a half-day's 13:00 close;
* the size threshold: |forecast| > `edge_bp` (and P(direction) ≥ `p_min` for a model that gives one);
* the model-health gate.

What passes all three are **the rule's decisions**. The book follows the position they ask for (§3), so a run of
same-side decisions, each extending the hold, is **one holding**: one entry and one exit, nothing dropped. The engine
makes one trade per holding, unless a stop closes it early: the holding's next decision then opens a new trade (a
fresh decision is a fresh trade), so a book with stops can make more trades than holdings.

**5b · the health gate.** Per session: the trailing rank correlation of the model's forecasts with the realized
returns over `health_days` sessions, from completed outcomes only, and the days the gate stood aside. The table
after it is the in-sample effect of the gate: the same rules with the gate off.

**5c · trade return by forecast size.** Do bigger forecasts make better trades? The trades are split into fifths by
the size of the forecast that opened them. Each fifth has four bars:

* the opening forecast (bp over the next `horizon` grid bars);
* the scored outcome over the trade's own grid bars: the realized close-to-close return of every grid bar from the
  opening decision to the bar the trade left in, signed as traded. For a trade held one hold, that is exactly what the
  model was scored on;
* the trade's return before costs and after costs.

The first two ask whether the forecast is right. The middle two ask whether the engine trades what was scored: a
trade before costs should sit close to its scored outcome; the gap is the stops and the one-minute fill delay. The
bars should rise with the forecast, and the largest fifth should clear the round trip. A trade extended by later
decisions spans several grid bars (`bars_held`), so its outcome adds up more than one forecast's worth.

**5d · one name on one day.** The price, every forecast against the `±edge_bp` threshold, and the trades it
produced, all on the New York clock. A forecast (bp over the hold) is drawn at its decision time, the close of its
grid bar; a trade on it fills at the next minute's open (one minute later when it switches sides), so its entry marker
sits at the same time. `SYMBOL` and `DAY` in §0 pick another one.

In [ ]:
rule = {k: PRM[k] for k in RULE}
edge_bp = fc["edge"] * inp["scale"] * 1e4  # forecast over the horizon in bp (trade_frames' units)
N1, M = panel.close.shape

print(
    f"shipped rule: edge_bp {rule['edge_bp']:g} bp, health_days {rule['health_days']}, "
    f"health_min {rule['health_min']:g}, hold {rule['hold_bars'] or PRM['horizon']} grid bar(s)"
)
fun = rs.funnel(rule, st, panel, fc, SCORE_FROM).to_frame("count")
fun["per year"] = (fun["count"] / st["years"]).round(0)
fun

In [ ]:
h = None
if rule["health_days"] > 0:
    hs = S["health"](fc, days=rule["health_days"])  # per session code
    day_of = pd.to_datetime(pd.Series(g["date"]).groupby(g["code"]).first())
    h = pd.Series(hs, index=day_of.reindex(range(len(hs))).to_numpy())
    h = h[h.index >= pd.Timestamp(SCORE_FROM)]
    traded = h > rule["health_min"]
    print(
        f"health gate open on {traded.mean():.0%} of {len(h)} scored sessions; stood aside on "
        f"{int((~traded).sum())} ({int(h.isna().sum())} of them without enough history)"
    )
    fig = go.Figure()
    for is_open, color, name in (
        (True, plots.SERIES[0], "gate open: traded"),
        (False, plots._MUTED, "gate closed: stood aside"),
    ):
        part = h[traded == is_open]
        fig.add_trace(
            go.Bar(
                x=part.index,
                y=part.fillna(0.0),
                name=name,
                marker_color=color,
                hovertemplate="%{x|%Y-%m-%d}<br>trailing rank IC %{y:+.3f}<extra></extra>",
            )
        )
    fig.add_hline(
        y=rule["health_min"],
        line={"color": plots._LOSS, "dash": "dot", "width": 1},
        annotation_text=f"health_min {rule['health_min']:g}",
        annotation_position="top left",
    )
    plots._base(
        fig,
        height=360,
        title=f"Fig 5b · the health gate: rank IC of the forecasts over the previous "
        f"{rule['health_days']} sessions",
    )
    fig.update_yaxes(title_text="trailing rank IC")
    charts.below(fig, -0.18).show()
    pf_off, st_off = scored({"health_days": 0})
    keep = ["alpha_cagr", "excess_cagr", "cagr", "volatility", "trades_per_year", "win_rate"]
    display(pd.DataFrame({"gate as shipped": st, "gate off": st_off}).loc[keep].round(4))
else:
    print("health_days is 0: the gate is off")

In [ ]:
row_of = pd.Series(np.arange(N1), index=panel.close.index)
bucket_of = np.searchsorted(g["start"], np.arange(N1), side="right") - 1  # 1-minute row -> grid bar
is_end = np.zeros(N1, bool)
is_end[g["end"]] = True  # the decision rows
r_in = row_of.loc[trades["entry_ts"]].to_numpy()
# the opening decision: the bar before the fill (two bars before when the trade switched sides)
dec = np.where(is_end[r_in - 1], bucket_of[r_in - 1], bucket_of[r_in - 2])
last = bucket_of[row_of.loc[trades["exit_ts"]].to_numpy() - 1]  # the last grid bar held
col = pd.Index(panel.symbols).get_indexer(trades["symbol"])
side = np.where(trades["direction"] == "Long", 1.0, -1.0)
cum = np.cumsum(np.nan_to_num(inp["y"] * inp["scale"]), axis=0)  # realized grid-bar log returns
trades["forecast_bp"] = edge_bp[dec, col]
trades["outcome_bp"] = side * 1e4 * (cum[last, col] - cum[dec, col])
trades["bars_held"] = last - dec
known = trades.dropna(subset=["forecast_bp"])
if len(known):
    print(
        f"{len(known)} trades, {known['bars_held'].mean():.1f} grid bars held on average "
        f"({(known['bars_held'] > (rule['hold_bars'] or PRM['horizon'])).mean():.0%} extended "
        f"by later decisions): {known['return_bp'].mean() + RT_BP:+.1f} bp before costs, against "
        f"{known['outcome_bp'].mean():+.1f} bp scored outcome over the same grid bars and "
        f"{known['forecast_bp'].abs().mean():.1f} bp opening forecast"
    )
if len(known) >= 50:
    fifth = pd.qcut(
        known["forecast_bp"].abs().rank(method="first"),
        5,
        labels=["1 smallest", "2", "3", "4", "5 largest"],
    )
    size = known.groupby(fifth, observed=True).agg(
        trades=("pnl", "size"),
        forecast_bp=("forecast_bp", lambda s: s.abs().mean()),
        outcome_bp=("outcome_bp", "mean"),
        net_bp=("return_bp", "mean"),
        bars_held=("bars_held", "mean"),
        win_rate=("pnl", lambda s: (s > 0).mean()),
    )
    size["before_costs_bp"] = size["net_bp"] + RT_BP
    size.index.name = "fifth of |forecast|"
    display(size.round(2))
    fig = go.Figure()
    for colname, color, name in (
        ("forecast_bp", plots.SERIES[4], "opening forecast (mean |bp|)"),
        ("outcome_bp", plots.SERIES[2], "scored outcome over the trade's grid bars"),
        ("before_costs_bp", plots.SERIES[0], "trade before costs"),
        ("net_bp", plots.SERIES[7], "trade after costs"),
    ):
        fig.add_trace(
            go.Bar(x=size.index.astype(str), y=size[colname], name=name, marker_color=color)
        )
    fig.add_hline(
        y=RT_BP,
        line={"color": plots._LOSS, "dash": "dot", "width": 1},
        annotation_text=f"one round trip {RT_BP:.1f} bp",
        annotation_position="top left",
    )
    fig.update_layout(barmode="group")
    plots._base(
        fig, title="Fig 5c · trade return by the size of the forecast behind it", height=380
    )
    fig.update_xaxes(title_text="fifth of |forecast|")
    fig.update_yaxes(title_text="bp per trade")
    charts.below(fig, -0.25).show()
else:
    print(f"only {len(known)} trades: too few to split by forecast size")

In [ ]:
sym = SYMBOL or (trades["symbol"].value_counts().index[0] if len(trades) else panel.symbols[0])
j = panel.symbols.index(sym)
mine = trades[trades["symbol"] == sym]
entry_day = pd.Series(rs.ny(mine["entry_ts"], panel.tz).date, index=mine.index)
day = (
    pd.Timestamp(DAY).date()
    if DAY
    else (entry_day.value_counts().index[0] if len(mine) else sess[-1])
)
px = panel.close[sym][rs.ny(panel.close.index, panel.tz).date == day]
b = np.flatnonzero(g["date"] == day)  # the day's grid bars
fcast = edge_bp[b, j]
today = mine[entry_day == day]
state = (
    ""
    if h is None
    else (
        " · health gate open"
        if bool(h.get(pd.Timestamp(day), np.nan) > rule["health_min"])
        else " · health gate closed"
    )
)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    row_heights=[0.62, 0.38],
    subplot_titles=(
        f"price and trades ({len(today)} trades)",
        f"forecast over the hold (bp) at each decision; threshold ±{rule['edge_bp']:g} bp",
    ),
)
fig.add_trace(
    go.Scatter(  # one session, 390 points: nothing to thin. Not plots._scatter: its epoch-ms x is
        # drawn in the viewer's LOCAL time, hours away from the trades' New York clock
        x=rs.ny(px.index, panel.tz),
        y=px.to_numpy(),
        mode="lines",
        name="close (1-minute)",
        line={"color": plots._INK, "width": 1.2},
        hovertemplate="%{x|%H:%M}<br>%{y:.2f}<extra>close</extra>",
    ),
    row=1,
    col=1,
)
for direction, marker, color in (
    ("Long", "triangle-up", plots._LONG),
    ("Short", "triangle-down", plots._SHORT),
):
    t = today[today["direction"] == direction]
    fig.add_trace(
        go.Scatter(
            x=rs.ny(t["entry_ts"], panel.tz),
            y=t["entry_price"],
            mode="markers",
            name=f"{direction.lower()} entry",
            marker={
                "symbol": marker,
                "size": 11,
                "color": color,
                "line": {"color": "white", "width": 1},
            },
        ),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(
        x=rs.ny(today["exit_ts"], panel.tz),
        y=today["exit_price"],
        mode="markers",
        name="exit",
        marker={"symbol": "x", "size": 8, "color": plots._MUTED},
        customdata=today[["return_bp"]].to_numpy(),
        hovertemplate="exit %{x|%H:%M} @ %{y:.2f}<br>%{customdata[0]:+.1f} bp net<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(  # at the decision: the close of the grid bar's last minute (stamped at its open)
        x=rs.ny(panel.close.index[g["end"][b]] + pd.Timedelta(minutes=1), panel.tz),
        y=fcast,
        name="forecast",
        showlegend=False,
        marker_color=np.where(fcast >= 0, plots._LONG, plots._SHORT),
        hovertemplate="%{x|%H:%M}<br>%{y:+.1f} bp<extra>forecast</extra>",
    ),
    row=2,
    col=1,
)
for level in (rule["edge_bp"], -rule["edge_bp"]):  # after the bars: shapes need a trace first
    fig.add_hline(y=level, line={"color": plots._MUTED, "dash": "dot", "width": 1}, row=2, col=1)
plots._base(fig, title=f"Fig 5d · {sym} on {day}{state}", height=620)
fig.update_xaxes(type="date")
fig.update_yaxes(title_text="bp", row=2, col=1)
charts.below(fig, -0.1).show()

## 6 · The study — the trading rule, searched in-process

`optimize.run_study` runs on the options-attached TUNE panel, in this process. Every trial re-scores the forecasts
cached in §1 (no refit), so a trial costs seconds.

* **One objective: `alpha_cagr`** (maximized), every trial scored warm from `score_from`. `edge_bp` and `health_min`
  are exposure knobs: scored against `volatility` as well, the trials would trade alpha for time in cash and the
  front would become an exposure dial (informed_flow v26). The volatility rides along with every trial.
* **Pruned:** trials trading less than `min_trades_per_year` (300 a year), so a handful of lucky fills cannot win.
  Forecasts above 10 bp are already rare (§5a), so on this forecast an `edge_bp` above ~15 bp or a `health_min`
  above ~0.04 trades too little: expect much of the box to be pruned (the cell prints the count).
* **Searched:** `edge_bp`, `hold_bars`, `stop_vol` and `health_min`, the knobs with an Optuna range in §2. The
  forecast config is fixed per study.
* **Journal:** `studies/<STUDY>.log`. Re-running this cell tops it up TO `N_TRIALS`, never past it.

Why not `tsim optimize`: a headless study has no option inputs (README, "Setup and caveats"), so it would score a
different model. For the same reason this cell refuses to run on a panel without them.

In [ ]:
if PRM["options"] != "none" and not inp["options_attached"]:
    raise RuntimeError(
        f"options={PRM['options']!r} but the panel carries no opt_* inputs: the study would score "
        "a model whose option inputs are all 0. Load the panel with research.load(options=True)."
    )
try:
    done = len(opt.load_study(run_strat, STUDY).trials)
except KeyError:  # no journal yet
    done = 0
todo = max(0, N_TRIALS - done)
print(f"{STUDY}: {done} trial(s) in the journal; running {todo}")
study = opt.run_study(
    run_strat,
    panel,
    settings=settings,
    study_name=STUDY,
    n_trials=todo,
    score_from=SCORE_FROM,
    verbose=True,
)
trials = opt.trials_frame(study)  # every completed trial (pruned: too few trades, or none)
states = pd.Series([t.state.name.lower() for t in study.trials]).value_counts()
print(f"{len(study.trials)} trials: " + ", ".join(f"{n} {s}" for s, n in states.items()))

## 7 · The pick and the candidates file (Table 7, Fig 7)

Only trials trading at least the pick floor (`optuna.pick_min_trades_per_year`, else `min_trades_per_year`) are
admissible. Among them `opt.pick_from_front`, the repo's one picking rule, takes the best `alpha_cagr`, never a trial
with negative alpha, and collapses tied outcomes first. With one objective the Pareto front is a single row, so the
rule chooses among **all** admissible trials: a best trial that trades too little falls back to the next one
instead of winning. When no trial has positive alpha the rule still returns the best one, flagged; this notebook then
calls it **"best trial (NOT admissible)"** from §9 on. That row is the study's best losing rule, kept as a control,
never a rule to trade.

* **Table 7** the distinct admissible outcomes, best first, the pick in bold. A row's `trial` number is how
  `darts_forecast_diagnostics.ipynb` names it as a candidate (`CANDIDATE = <trial>`, or `"pick"`).
* **Fig 7** alpha against each searched knob, one dot per completed trial; the star is the pick, the dashed line
  `config.yaml`'s value. **Read the shape before the peak.** A flat cloud says the knob hardly matters on TUNE and
  the pick is a coin flip between neighbours; a lone peak is the warning sign of fitted luck. When every trial loses,
  look for the third shape: alpha climbing toward zero as the rule trades less (fewer round trips to pay) until the
  trade floor stops it. A pick where the cloud meets the floor is the cheapest way to trade noise, not an edge. All of
  it is TUNE: the study chose these knobs on exactly these months.

The best distinct admissible trials (up to ten) are saved to `studies/<STUDY>.candidates.yaml` (git-ignored) with
the forecast config they trade, the pick, the scored window and whether the pick is admissible. The diagnostics
notebook reads that file.

In [ ]:
SPACE = strat.optuna["space"]
HURDLE = OBJECTIVES[0]  # alpha_cagr

best_row, picked, pool, menu = None, {}, trials, trials
if trials.empty:
    admissible = trials
    print(
        "!! every trial was pruned (too few trades or none): nothing to pick; §9-§12 run the "
        "shipped rules only"
    )
else:
    # what the rule picks among: the Pareto front, or with one objective every completed trial
    front = opt.pareto_frame(study) if len(OBJECTIVES) > 1 else trials
    admissible = front[front["trades_per_year"] >= FLOOR]
    pool = admissible if len(admissible) else front
    if admissible.empty:
        print(f"!! no trial trades >= {FLOOR}/yr: picking from all {len(front)}")
    best_row = opt.pick_from_front(pool, objectives=OBJECTIVES)
    picked = rs.as_params(best_row, SPACE)
    print(
        f"{len(study.trials)} trials, {len(trials)} completed, {len(admissible)} admissible "
        f"(>= {FLOOR} trades/yr), {int(best_row['front_distinct'])} distinct outcomes; pick: trial "
        f"{int(best_row['trial'])}, {HURDLE} {best_row[HURDLE]:+.2%}/yr at "
        f"{best_row['trades_per_year']:,.0f} trades/yr"
        + (
            f" (tied with {int(best_row['n_tied']) - 1} identical-scoring trial(s))"
            if best_row["n_tied"] > 1
            else ""
        )
        + ("" if best_row["beats_benchmark"] else f" -- NO trial has {HURDLE} > 0")
    )
    order = trials.sort_values("trial")[HURDLE].cummax()
    third = order.iloc[-1] - order.iloc[max(0, int(len(order) * 2 / 3) - 1)]
    print(
        f"convergence: the last third of the completed trials raised the best {HURDLE} by "
        f"{third:+.2%}: "
        + ("converged" if third < 0.002 else "still improving, so the pick is not an optimum yet")
    )
    menu = opt.distinct_front(pool, objectives=OBJECTIVES).sort_values(HURDLE, ascending=False)
    menu = menu.head(10)
    cols = [
        c
        for c in (
            "trial",
            *SPACE,
            "alpha_cagr",
            "volatility",
            "beta",
            "excess_cagr",
            "max_drawdown",
            "trades_per_year",
        )
        if c in menu
    ]
    pick_no = int(best_row["trial"])
    fmt = {k: "{:.4g}" for k, spec in SPACE.items() if spec["type"] != "categorical"}
    fmt |= {
        "alpha_cagr": "{:+.1%}",
        "excess_cagr": "{:+.1%}",
        "volatility": "{:.1%}",
        "max_drawdown": "{:+.1%}",
        "beta": "{:+.3f}",
        "trades_per_year": "{:,.0f}",
    }  # ONE format call: a second .format() resets the columns the first one set
    display(
        menu[cols]
        .style.hide(axis="index")
        .format(fmt)
        .apply(lambda r: ["font-weight: bold" if r["trial"] == pick_no else ""] * len(r), axis=1)
    )
    knobs = list(SPACE)
    fig = make_subplots(
        rows=1, cols=len(knobs), shared_yaxes=True, horizontal_spacing=0.03, subplot_titles=knobs
    )
    for i, k in enumerate(knobs, start=1):
        fig.add_trace(
            go.Scatter(
                x=trials[k],
                y=trials[HURDLE],
                mode="markers",
                name="completed trial",
                legendgroup="trial",
                showlegend=i == 1,
                marker={"color": plots.SERIES[0], "size": 7, "opacity": 0.55},
                customdata=trials[["trial", "trades_per_year"]].to_numpy(),
                hovertemplate=f"trial %{{customdata[0]}}<br>{k} %{{x}}<br>{HURDLE} %{{y:+.1%}}"
                "<br>%{customdata[1]:,.0f} trades/yr<extra></extra>",
            ),
            row=1,
            col=i,
        )
        fig.add_trace(
            go.Scatter(
                x=[best_row[k]],
                y=[best_row[HURDLE]],
                mode="markers",
                name=f"pick: trial {pick_no}",
                legendgroup="pick",
                showlegend=i == 1,
                marker={
                    "symbol": "star",
                    "size": 16,
                    "color": plots.SERIES[7],
                    "line": {"color": "white", "width": 1},
                },
                hovertemplate=f"pick<br>{k} %{{x}}<br>{HURDLE} %{{y:+.1%}}<extra></extra>",
            ),
            row=1,
            col=i,
        )
    for i, k in enumerate(knobs, start=1):  # after the traces: a shape needs one in its subplot
        fig.add_vline(
            x=PRM[k], line={"color": plots._MUTED, "dash": "dash", "width": 1}, row=1, col=i
        )
        fig.add_hline(y=0, line={"color": plots._LOSS, "dash": "dot", "width": 1}, row=1, col=i)
    plots._base(
        fig,
        title=f"Fig 7 · {HURDLE} (TUNE) against each searched knob; dashed: config.yaml's value",
        height=400,
    )
    for i, k in enumerate(knobs, start=1):  # each x axis spans the searched range
        spec = SPACE[k]
        if spec["type"] != "categorical":
            pad = 0.04 * (spec["high"] - spec["low"])
            fig.update_xaxes(range=[spec["low"] - pad, spec["high"] + pad], row=1, col=i)
        elif all(isinstance(c, int | float) for c in spec["choices"]):
            fig.update_xaxes(
                range=[min(spec["choices"]) - 0.5, max(spec["choices"]) + 0.5],
                tickvals=list(spec["choices"]),
                row=1,
                col=i,
            )
    fig.update_layout(margin={"t": 90})
    fig.update_yaxes(tickformat="+.0%")
    fig.update_yaxes(title_text=f"{HURDLE} (TUNE)", row=1, col=1)
    charts.below(fig, -0.2).show()

ok_pick = best_row is not None and len(admissible) > 0 and bool(best_row["beats_benchmark"])
PICK = "picked rules" if ok_pick else "best trial (NOT admissible)"  # its name from §9 on
if picked:
    print(f"{PICK}:", picked)
METRICS = (
    "alpha_cagr",
    "volatility",
    "beta",
    "excess_cagr",
    "info_ratio",
    "cagr",
    "max_drawdown",
    "trades_per_year",
    "years",
)
snapshot = {
    "study": STUDY,
    "saved": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "forecast": {k: PRM[k] for k in S["MODEL_KEYS"]},  # the fixed forecast these rules trade
    "pick_min_trades_per_year": int(FLOOR),
    "score_from": SCORE_FROM,
    "admissible": ok_pick,
    "pick": None if best_row is None else int(best_row["trial"]),
    "front": [
        {
            "trial": int(r["trial"]),
            **{m: round(float(r[m]), 6) for m in METRICS if m in r},
            "params": rs.as_params(r, SPACE),
        }
        for _, r in (menu.iterrows() if best_row is not None else iter(()))
    ],
}
cand_path = opt.study_path(run_strat, STUDY).with_name(f"{STUDY}.candidates.yaml")
cand_path.write_text(yaml.safe_dump(snapshot, sort_keys=False))
print(f"saved {len(snapshot['front'])} candidate(s) -> studies/{cand_path.name}")

## 8 · Boundary check — did the pick hit the edge of the searched range?

A picked value on the edge of its range, or half or more of the distinct admissible outcomes piled onto one edge,
hints that the optimum lies outside it. If a parameter is flagged:

1. widen that bound in `config.yaml`;
2. bump `optuna.study` (a journal that mixes two spaces cannot be read);
3. re-run §6 and re-read §7–§8.

Read Fig 7 first: on a flat cloud an edge pick is noise, not a direction to search in. Categorical knobs
(`hold_bars`) have no edges and are skipped.

In [ ]:
if best_row is None:
    print("no pick: nothing to check")
else:
    report = opt.boundary_report(picked, SPACE, front=pool)
    flagged = report[report["action"] != "ok"]
    print(
        "all clear: the searched ranges bracket the pick"
        if flagged.empty
        else f"{len(flagged)} parameter(s) pressing against the search space:"
    )
    display(report)

## 9 · The picked rules against the shipped rules and the baseline — TUNE (Table 9a, Table 9b, Fig 9)

The pick re-run as a full backtest on the same scored TUNE window (named "best trial (NOT admissible)" when §7 found
no admissible pick). The study chose these knobs on exactly these months, so a better number here is expected and
proves nothing by itself. The test is TEST, read once: `darts_forecast_diagnostics.ipynb` with `CANDIDATE = "pick"`
(or §11 here, with `READ_TEST = True`). **Table 9b** is §5a's funnel for both rules: a lower `edge_bp` admits smaller
forecasts, so more decisions and longer holdings.

In [ ]:
tested = {"shipped rules": {}, **({PICK: picked} if picked else {})}  # §9-§12
books, stats_tune = {"shipped rules": pf}, {"shipped rules": st}
funnels = {"shipped rules": rs.funnel(rule, st, panel, fc, SCORE_FROM)}
if picked:
    books[PICK], stats_tune[PICK] = scored(picked)
    funnels[PICK] = rs.funnel({**rule, **picked}, stats_tune[PICK], panel, fc, SCORE_FROM)
stats_tune["baseline (buy & hold)"] = st_base
display(pd.DataFrame(stats_tune).loc[COLS].round(4))
display(pd.DataFrame(funnels))
charts.overlay(
    books,
    f"Fig 9 · TUNE from {SCORE_FROM}: shipped rules, the study's row, buy-and-hold",
    pf_base,
    INIT_CASH,
).show()

## 10 · Does retraining pay? The cadence, traded — TUNE (Table 10, Fig 10)

The same model and the same trading rules, on forecasts retrained at different cadences (`CADENCES` in §0, in
trading minutes):

* `0` trains once, on the first TRAIN window, and never again;
* `7800` retrains every 20 sessions, `1950` weekly;
* `390` retrains before every session, the shipped default.

Each cadence is its own walk-forward, cached like §1's. The model notebook runs the same cadences on forecast
quality alone. How to read Table 10:

1. **The IC columns first.** Retraining can only help a trading rule by helping the forecast. An IC with |t| below
   2 cannot be told apart from zero, and a cadence whose IC is no better than training once is not adapting to
   anything: its alpha differences are noise.
2. **Then the trade counts.** `edge_bp` is a fixed threshold, but each cadence's forecasts have their own size, so
   the same rules trade much more on one cadence than on another. A cadence that hardly trades has an alpha near
   zero because it was hardly in the market, not because it was better.

This is TUNE: it informs which forecast config deserves a study, it is not a verdict.

In [ ]:
rows, curves = [], {}
for c in CADENCES:
    p_c = {**PRM, "retrain_minutes": c}
    fc_c = rs.prime_cache(panel, p_c)  # cached per cadence, shared with the model notebook
    skill_c = rs.summary(rs.table(fc_c, (SCORE_FROM, TUNE[1])))
    row = {
        "retrain_minutes": c,
        "cadence": rs.cadence_name(c) + (" (this run)" if c == PRM["retrain_minutes"] else ""),
        "fits": len(fc_c["log"]),
        "fit minutes": fc_c["log"]["seconds"].sum() / 60,
        "rank IC": skill_c["ic"],
        "IC t": skill_c["t"],
        "hit": skill_c["hit"],
    }
    for label, params in tested.items():
        book_c, st_c = scored(params, strategy=replace(strat, params=p_c))
        row[f"alpha · {label}"] = st_c["alpha_cagr"]
        row[f"trades/yr · {label}"] = st_c["trades_per_year"]
        curves.setdefault(label, {})[c] = book_c
    rows.append(row)
cadence = pd.DataFrame(rows).set_index("retrain_minutes")
display(cadence.round(4))
fig = go.Figure()
for i, (c, book_c) in enumerate(curves["shipped rules"].items()):
    curve = bt.equity_curve(book_c, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0
    fig.add_trace(
        plots._scatter(
            curve,
            mode="lines",
            name=f"retrained {rs.cadence_name(c)}",
            line={"color": plots.SERIES[i], "width": 2 if c == PRM["retrain_minutes"] else 1.2},
        )
    )
plots._base(
    fig,
    title="Fig 10 · the shipped rules by retraining cadence: cumulative return (TUNE)",
    height=420,
)
fig.update_xaxes(type="date")
fig.update_yaxes(tickformat="+.1%", title_text="cumulative return")
charts.below(fig).show()

## 11 · The TEST replay — only with `READ_TEST = True` (Table 11a, Table 11b, Fig 11)

**Off by default.** TEST (2026-02-01 → 2026-04-30) is read once, in `darts_forecast_diagnostics.ipynb`: the frozen
config, or a row of §7's candidates file, on TEST with its stress tests, the coin-flip null and the verdict. With
`READ_TEST = True` this section replays the shipped rules and the pick of §7 on TEST:

* **Warm.** The panel is loaded from `data.start` through the last TEST session, and the walk-forward simply continues
  with the same retraining: every TEST forecast comes from a model fit only on sessions before it. The book and the
  metrics start at TEST's first bar (`score_from`). The TEST forecasts are cached under their own key (this panel's
  content), shared with the diagnostics notebook.
* **TEST's one look.** If anything is changed BECAUSE of these numbers, TEST has become TUNE data and the claim must
  move to the HOLDOUT. Re-running the cell is not a second look (the same forecasts give the same numbers), and neither
  is the diagnostics notebook reporting the same frozen config; changing a knob and re-running is.
* **The HOLDOUT is untouched.** 2026-05-01 onward is never loaded here.

**Table 11a** the books against buy-and-hold over TEST. **Table 11b** the forecaster's own skill on scored TUNE and
on TEST (rank IC with a day-clustered t, hit rate, AUC; `research.summary`). A trading result on TEST that the
forecast's IC does not support is luck.

In [ ]:
panel_test = fc_test = last_test = skill = None
books_test, stats_test = {}, {}
if not READ_TEST:
    print(
        "READ_TEST = False: TEST is not loaded here. Its one-shot read (the frozen config, the "
        "stress tests, the coin-flip null, the verdict) lives in darts_forecast_diagnostics.ipynb; "
        "set READ_TEST = True in §0 to replay the shipped and picked rules here."
    )
else:
    panel_test = rs.load(settings, engine, end=TEST[1])  # data.start -> the last TEST session
    last_test = panel_test.close.index[-1].tz_convert(panel_test.tz).date()
    assert str(last_test) <= TEST[1], "the TEST panel reaches into the HOLDOUT"
    fc_test = rs.prime_cache(panel_test, PRM)  # the same forecast config, continued through TEST
    for label, params in tested.items():
        books_test[label], stats_test[label] = scored(params, p=panel_test, start=TEST[0])
    pf_base_test, stats_test["baseline (buy & hold)"] = scored(
        {}, strategy=baseline, p=panel_test, start=TEST[0]
    )
    print(
        f"TEST panel: {len(panel_test.close):,} bars, last session {last_test}; "
        f"scored from {TEST[0]}"
    )
    display(pd.DataFrame(stats_test).loc[COLS].round(4))
    skill = pd.DataFrame(
        {
            "scored TUNE": sk,  # §1
            "TEST": rs.summary(rs.table(fc_test, "test")),
        }
    ).T
    display(skill[["ic", "t", "hit", "auc", "mse_skill", "sign_bp", "n", "days"]].round(4))
    charts.overlay(
        books_test,
        f"Fig 11 · TEST {TEST[0]} → {last_test}: the frozen rules vs buy-and-hold",
        pf_base_test,
        INIT_CASH,
    ).show()

## 12 · Cost stress (Table 12)

The same books, re-priced on TUNE (and on TEST with `READ_TEST = True`). `bt.active_stats` re-prices the
buy-and-hold hurdle at the same costs, so the comparison stays fair.

* **As configured:** 14 bp per round trip at the \$2,000 order size (`research.round_trip_bp`: the \$1 minimum
  commission is 5 bp per leg, plus half the spread and the slippage).
* **2× every cost:** commission, minimum commission, spread and slippage all doubled.
* **`COST_BARS`:** a fixed quoted spread (half of it charged per leg) and slippage per leg, in bp; 10/3 is the
  repo's usual "volatile names" bar.
* **Pessimistic fills:** every buy at the bar's high and every sell at its low, the benchmark included.

A 5-minute forecast has to clear one round trip on every trade. If alpha flips sign at 2× costs, the edge is thinner
than the costs are uncertain. The round-trip column is the engine's cost model only; pessimistic fills add each fill
bar's half-range on top.

In [ ]:
cases = {
    "as configured": (run_strat, settings),
    "2x every cost": (run_strat, rs.with_costs(settings, 2.0)),
}
for spread, slippage in COST_BARS:
    cases[f"{spread:g}/{slippage:g} bp per leg"] = (
        run_strat,
        rs.with_costs(settings, spread=spread, slippage=slippage),
    )
cases["pessimistic fills"] = (replace(run_strat, fill="pessimistic"), settings)
windows = [("TUNE", panel, SCORE_FROM)] + ([("TEST", panel_test, TEST[0])] if READ_TEST else [])
rows = []
for case, (strategy, sett) in cases.items():
    for label, params in tested.items():
        for window, p, start in windows:
            _, x = scored(params, strategy=strategy, p=p, start=start, sett=sett)
            rows.append(
                {
                    "case": case,
                    "rules": label,
                    "window": window,
                    "round trip bp": rs.round_trip_bp(sett, p),  # the engine's cost model
                    **{m: x[m] for m in ("alpha_cagr", "excess_cagr", "cagr", "trades_per_year")},
                }
            )
stress = pd.DataFrame(rows).set_index(["case", "rules", "window"])
stress.round(4)

## 13 · Checks and the scorecard

The checks are about plumbing and protocol, not about whether the strategy works:

* the data split was respected, and the option inputs the model reads were attached;
* the forecasts are out of sample: every forecast comes from a fit made at or before its decision bar, and every
  TRAIN window is the `train_days` sessions before its retrain point;
* the cached walk-forward belongs to this panel: the panel id stored with it equals this panel's content fingerprint;
* §5a's gates reproduce `strategy.trade_frames`' position path, for the shipped rules and the pick: every entry sits
  on the fill bar of a decision of its side, and every decision's side is held from its fill bar;
* the backtest helper agrees with `bt.evaluate`, and the baseline ran on the same window;
* the study holds what it should.

One row is a result, not plumbing: whether the study found an admissible rule with positive alpha. With
`READ_TEST = True`, three more rows check that appending TEST changes no TUNE forecast (anything that read the future
would move them), that the TEST forecasts belong to the TEST panel and that the TEST panel stops at the TEST end.
Without it, the diagnostics notebook runs those checks when it reads TEST (its §2).

The `SCORECARD` line is the machine-readable summary of the run.

In [ ]:
ev = bt.evaluate(run_strat, panel, {}, settings=settings, score_from=SCORE_FROM)
cells = np.isfinite(fc["edge"])
made_by = log.set_index("fit")["retrain_bucket"].reindex(fc["fit"][cells].astype(int)).to_numpy()
fit_ok = bool((made_by <= np.nonzero(cells)[0]).all())  # fit k only forecasts from its own point on
S_IDX = {str(d): i for i, d in enumerate(sess)}  # session date -> its number
val_n = PRM["val_days"] if PRM["model"] in rs.SM["DEEP"] else 0
n_window = log["session"].map(S_IDX) - log["train_from"].map(S_IDX)  # sessions before the retrain
gap = log["session"].map(S_IDX) - log["train_last"].map(S_IDX)  # last TRAIN session -> retrain
opens = g["first"][log["retrain_bucket"].to_numpy()]  # the retrain point opens its session
train_ok = bool((n_window == PRM["train_days"]).all()) and bool(
    (gap.to_numpy() >= np.where(opens, 1 + val_n, val_n)).all()
)
misses = sum(  # §5a's gates against trade_frames' position path: the shipped rules and the pick
    ck.path_misses(rs.decisions(r, fc), S["trade_frames"](panel, fc, **r), g["end"], rs.held)
    for r in ([rule, {**rule, **picked}] if picked else [rule])
)
trial_floor = strat.optuna["min_trades_per_year"]
checks = [
    ("the study panel ends inside TUNE", str(sess[-1]) <= TUNE[1], f"last session {sess[-1]}"),
    (
        "the option inputs the model reads are attached",
        PRM["options"] == "none" or bool(inp["options_attached"]),
        f"options={PRM['options']!r}, {len(OPT_FRAMES)} opt_* frames on the panel",
    ),
    (
        "scoring starts at the first fitted model",
        log["session"].iloc[0] <= SCORE_FROM,
        f"score_from {SCORE_FROM}, first fit {log['session'].iloc[0]}",
    ),
    (
        "every forecast comes from a fit made at or before its decision bar",
        fit_ok,
        f"{int(cells.sum()):,} forecasts, {len(log)} fits",
    ),
    (
        "every TRAIN window is the train_days sessions before its retrain point",
        train_ok,
        f"{n_window.min()}-{n_window.max()} sessions (train_days {PRM['train_days']}); the last "
        f"TRAIN session {gap.min()}-{gap.max()} session(s) before the retrain",
    ),
    (
        "the cached walk-forward belongs to this panel (content fingerprint)",
        fc["panel_id"] == S["fingerprint"](panel),
        f"cache/{rs._key(PRM, panel)}.pkl",
    ),
    (
        "§5a's gates reproduce trade_frames' position path",
        misses == 0,
        f"{misses} disagreement(s), shipped rules" + (" and the pick" if picked else ""),
    ),
    (
        "bt.evaluate reproduces §3",
        abs(ev["alpha_cagr"] - st["alpha_cagr"]) < 1e-9,
        f"alpha {ev['alpha_cagr']:+.4f} vs {st['alpha_cagr']:+.4f}",
    ),
    (
        "the baseline is scored on the same window",
        abs(st_base["years"] - st["years"]) < 1e-9,
        f"{st_base['years']:.3f} vs {st['years']:.3f} years",
    ),
    (
        "the shipped rules clear min_trades_per_year",
        st["trades_per_year"] >= trial_floor,
        f"{st['trades_per_year']:,.0f}/yr vs {trial_floor}/yr",
    ),
    (
        "the study journal holds N_TRIALS trials",
        len(study.trials) >= N_TRIALS,
        f"{len(study.trials)} of {N_TRIALS} ({len(trials)} completed)",
    ),
    (
        "RESULT: an admissible pick with positive alpha",
        bool(snapshot["admissible"]),
        "none" if best_row is None else f"trial {int(best_row['trial'])}",
    ),
]
if READ_TEST:
    n_tune = len(fc["edge"])  # the TUNE grid is a prefix of the TEST panel's grid
    both = np.isfinite(fc["edge"]) & np.isfinite(fc_test["edge"][:n_tune])
    same = np.array_equal(fc["edge"][both], fc_test["edge"][:n_tune][both])
    checks += [
        (
            "appending TEST changes no TUNE forecast",
            same and both.sum() == np.isfinite(fc["edge"]).sum(),
            f"{int(both.sum()):,} forecasts compared",
        ),
        (
            "the cached TEST walk-forward belongs to the TEST panel",
            fc_test["panel_id"] == S["fingerprint"](panel_test),
            f"cache/{rs._key(PRM, panel_test)}.pkl",
        ),
        (
            "the TEST panel stops at the TEST end (HOLDOUT unread)",
            str(last_test) <= TEST[1],
            f"last session {last_test}",
        ),
    ]
else:
    checks.append(
        (
            "TEST not loaded here (READ_TEST = False)",
            panel_test is None,
            "its one-shot read and the truncation check: darts_forecast_diagnostics.ipynb",
        )
    )
CHECKS = rs.checks_table(checks)
with pd.option_context("display.max_colwidth", None):
    display(CHECKS)

KEY = {"shipped rules": "shipped", PICK: "picked", "baseline (buy & hold)": "baseline"}  # stable
rs.scorecard(
    {
        "notebook": "strategy",
        "study": STUDY,
        "forecast": {k: PRM[k] for k in ("model", "bar_minutes", "horizon", "retrain_minutes")},
        "options_attached": bool(inp["options_attached"]),
        "score_from": SCORE_FROM,
        "round_trip_bp": RT_BP,
        "tune_ic": sk["ic"],
        "tune_ic_t": sk["t"],
        "trials": len(study.trials),
        "completed": len(trials),
        "pick": snapshot["pick"],
        "pick_admissible": snapshot["admissible"],
        "picked": picked,
        "tune": {KEY[k]: rs.book(v) for k, v in stats_tune.items()},
        "cadence_ic": {int(c): round(float(v), 4) for c, v in cadence["rank IC"].items()},
        "read_test": READ_TEST,
        "test": {KEY[k]: rs.book(v) for k, v in stats_test.items()} if READ_TEST else None,
        "test_ic": None if skill is None else skill.loc["TEST", "ic"],
        "checks_failed": [c for c, ok, _ in checks if not ok],
    }
)

## Read the result honestly

* **What is in sample.** The forecasts are out of sample: each comes from a model fit only on earlier sessions. The
  trading rule in §6–§9 is not: the study chose `edge_bp`, the hold, the stop and the gate on the very months it is
  scored on. So did the choice of forecast config, in the model notebook. Only TEST is out of sample for the rule:
  `darts_forecast_diagnostics.ipynb`, or §11 here with `READ_TEST = True`.
* **The forecast comes first.** §1 prints the scored forecasts' rank IC and hit rate; the model notebook judges them
  against its coin-flip null. Searching a trading rule on top of a forecast without skill finds the luck of the TUNE
  window, or, when every trial loses, the rule that trades least (Fig 7). On TEST, read the forecast's IC before the
  book's alpha.
* **Alpha, not excess.** A book that is flat overnight and most of the day has a beta near zero. Losing to
  buy-and-hold in a rising market is expected; losing money after the beta it ran is the failure.
* **Costs decide a 5-minute strategy.** Every trade pays the round trip printed in §3 (14 bp at the \$2,000 order
  size). Fig 5c shows whether the forecasts that open trades are large and accurate enough to pay it. §12 shows how
  much margin is left.
* **Forecast or execution?** §5c's scored outcome is what the model was graded on, over the very grid bars the book
  held. When the trades before costs track it, the engine trades the forecast faithfully and a loss is the
  forecast's. When they do not, look at the stops and the fills first.
* **What would change the verdict.** An input with real information, such as order flow (IDEAS.md). It would be
  tested the same way: model notebook first, then this notebook, then one look at TEST.